# Amazon ML Challenge 2026: Business Entity Resolution
### Dual Tesla T4 (GPU T4 ×2) Optimized Reproducible Pipeline (Target: Macro F0.5 ~0.984+)

This notebook executes the complete 10-stage Business Entity Resolution pipeline on a Kaggle Notebook equipped with **Dual Tesla T4 GPUs (`GPU T4 ×2`, 32 GB total VRAM, 4 vCPUs)** within **~2 to 2.5 hours** (cleanly avoiding Kaggle's 12-hour session timeout).

#### Architectural Guarantees & Multi-GPU Optimizations:
1. **Dual-GPU DataParallel Acceleration**: Distributes heavy embedding (12.1 million records) and pair scoring across both `cuda:0` and `cuda:1` in parallel, cutting neural forward pass time by 50%.
2. **4-vCPU Parallelism**: Fully utilizes all 4 vCPUs for `rapidfuzz` string metrics and Polars normalizations in `features.py` and `features2.py`.
3. **Cross-Encoder `--used-only`**: Skips scoring Folds 5–9 on the training split (which the LightGBM ranker never reads), saving ~10 million unnecessary pair evaluations on the train set while scoring 100% of the test set.
4. **Tesla T4 Native FP16 Tensor Cores**: Uses `torch.float16` with `torch.cuda.amp.GradScaler` (bypassing slow software-emulated `bfloat16`).
5. **Bi-Encoder 500k Pair Convergence**: Capped to 500,000 pairs (~1,950 steps, convergence reached by step 200), cutting training from 10.5 hours to ~12 minutes.
6. **Exact Mathematical Formulation Preserved**: 49 Stage-1 features + 26 active Decoy features (87 total features in the final LightGBM ranker).
7. **Calibrated Decision Rule**: Greedy best-per-query decoding at calibrated threshold $t = 0.75$.
8. **Official Deliverables**: Generates both individual TSVs (`matching_results.tsv`, `candidate_pairs.tsv`) with exact headers, and builds the verified `final_submission.zip` package.

In [1]:
# STEP 1: HARDWARE & ENVIRONMENT DIAGNOSTICS (Dual T4 x2 & 4 vCPUs)
import os
import sys
import time
import subprocess
import torch

print('=' * 70)
print('STEP 1: HARDWARE & ENVIRONMENT DIAGNOSTICS')
print('=' * 70)

print(f'Python Version: {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
cuda_available = torch.cuda.is_available()
print(f'CUDA Available: {cuda_available}')

if cuda_available:
    device_count = torch.cuda.device_count()
    print(f'Detected CUDA GPUs: {device_count}')
    total_vram_gb = 0
    for i in range(device_count):
        free_mem, total_mem = torch.cuda.mem_get_info(i)
        total_vram_gb += total_mem / 2**30
        print(f'  [GPU {i}] {torch.cuda.get_device_name(i)} | VRAM: Free {free_mem / 2**30:.2f} GiB / Total {total_mem / 2**30:.2f} GiB')
    print(f'Total Combined VRAM: {total_vram_gb:.1f} GiB')
    
    # Benchmark FP16 Tensor Cores
    a = torch.randn(8192, 8192, device='cuda', dtype=torch.float16)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(20):
        b = a @ a
    torch.cuda.synchronize()
    t_fp16 = time.time() - t0
    tflops = (20 * 2 * (8192 ** 3)) / t_fp16 / 1e12
    print(f'Tesla T4 FP16 Tensor Core Speed: {tflops:.1f} TFLOPS (Hardware acceleration verified)')
    del a, b
    torch.cuda.empty_cache()
else:
    print('WARNING: CUDA not detected! Please enable GPU T4 x2 accelerator in Kaggle Notebook settings.')

cpus = os.cpu_count() or 1
print(f'Available CPU Cores: {cpus} vCPUs')

STEP 1: HARDWARE & ENVIRONMENT DIAGNOSTICS
Python Version: 3.12.13
PyTorch Version: 2.10.0+cu128
CUDA Available: True
Detected CUDA GPUs: 2
  [GPU 0] Tesla T4 | VRAM: Free 14.46 GiB / Total 14.56 GiB
  [GPU 1] Tesla T4 | VRAM: Free 14.46 GiB / Total 14.56 GiB
Total Combined VRAM: 29.1 GiB
Tesla T4 FP16 Tensor Core Speed: 17.4 TFLOPS (Hardware acceleration verified)
Available CPU Cores: 4 vCPUs


In [2]:
# STEP 2: DEPENDENCIES VERIFICATION & SETUP
print('=' * 70)
print('STEP 2: DEPENDENCIES VERIFICATION & SETUP')
print('=' * 70)

required_packages = [
    'polars',
    'rapidfuzz',
    'unidecode',
    'transformers',
    'lightgbm',
    'narwhals'
]

missing = []
for pkg in required_packages:
    try:
        __import__(pkg)
        print(f'  [OK] {pkg}')
    except ImportError:
        missing.append(pkg)
        print(f'  [MISSING] {pkg}')

if missing:
    print(f'\nInstalling missing packages: {missing}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    print('Dependencies installed successfully!')
else:
    print('All core dependencies are present.')

STEP 2: DEPENDENCIES VERIFICATION & SETUP
  [OK] polars
  [MISSING] rapidfuzz
  [MISSING] unidecode
  [OK] transformers
  [OK] lightgbm
  [OK] narwhals

Installing missing packages: ['rapidfuzz', 'unidecode']...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 10.1 MB/s eta 0:00:00
Dependencies installed successfully!


In [3]:
# STEP 3: AUTOMATIC DATASET DISCOVERY (Supports Structured & Flat Datasets)
print('=' * 70)
print('STEP 3: AUTOMATIC DATASET DISCOVERY')
print('=' * 70)

def discover_dataset(search_roots=None):
    if search_roots is None:
        search_roots = ['/kaggle/input', './student_resource/dataset', './dataset', '.']
    
    dataset_dir = None
    validator_path = None
    doc_template_path = None
    
    # Priority 1: Check for structured directory (train/ and test/ subfolders)
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            has_train = os.path.exists(os.path.join(dirpath, 'train', 'train_source1.tsv'))
            has_test = os.path.exists(os.path.join(dirpath, 'test', 'test_source1.tsv'))
            if has_train and has_test and dataset_dir is None:
                dataset_dir = os.path.abspath(dirpath)
            if 'validate_submission.py' in filenames and validator_path is None:
                validator_path = os.path.abspath(os.path.join(dirpath, 'validate_submission.py'))
            if 'Documentation_template.md' in filenames and doc_template_path is None:
                doc_template_path = os.path.abspath(os.path.join(dirpath, 'Documentation_template.md'))

    # Priority 2: Fallback to flat directory (clean_dataset) if structured not found
    if dataset_dir is None:
        for root in search_roots:
            if not os.path.exists(root):
                continue
            for dirpath, dirnames, filenames in os.walk(root):
                has_train = 'train_source1.tsv' in filenames and 'train_ground_truth.tsv' in filenames
                has_test = 'test_source1.tsv' in filenames
                if has_train and has_test and dataset_dir is None:
                    dataset_dir = os.path.abspath(dirpath)
                if 'validate_submission.py' in filenames and validator_path is None:
                    validator_path = os.path.abspath(os.path.join(dirpath, 'validate_submission.py'))
                if 'Documentation_template.md' in filenames and doc_template_path is None:
                    doc_template_path = os.path.abspath(os.path.join(dirpath, 'Documentation_template.md'))

    return dataset_dir, validator_path, doc_template_path

DATASET_DIR, VALIDATOR_PATH, DOC_TEMPLATE = discover_dataset()

print(f'Discovered Dataset Directory: {DATASET_DIR}')
print(f'Discovered Validator Script:  {VALIDATOR_PATH}')
print(f'Discovered Doc Template:      {DOC_TEMPLATE}')

assert DATASET_DIR is not None, 'ERROR: Could not locate dataset folder containing train/test TSVs!'

# Validate TSV files
for split in ['train', 'test']:
    for f in [f'{split}_source1.tsv', f'{split}_source2.tsv', f'{split}_source3.tsv']:
        fp = os.path.join(DATASET_DIR, split, f)
        if not os.path.exists(fp):
            fp = os.path.join(DATASET_DIR, f)
        if os.path.exists(fp):
            sz_mb = os.path.getsize(fp) / 2**20
            print(f'  [Found] {f} ({sz_mb:.1f} MB)')
        else:
            print(f'  [Missing!] {f}')

STEP 3: AUTOMATIC DATASET DISCOVERY
Discovered Dataset Directory: /kaggle/input/datasets/iamkamaltripathi/amazon-ml-clean-dataset-zip/dataset/student_resource/dataset
Discovered Validator Script:  /kaggle/input/datasets/iamkamaltripathi/amazon-ml-clean-dataset-zip/dataset/student_resource/utils/validate_submission.py
Discovered Doc Template:      /kaggle/input/datasets/iamkamaltripathi/amazon-ml-clean-dataset-zip/dataset/student_resource/Documentation_template.md
  [Found] train_source1.tsv (200.3 MB)
  [Found] train_source2.tsv (466.6 MB)
  [Found] train_source3.tsv (480.4 MB)
  [Found] test_source1.tsv (166.9 MB)
  [Found] test_source2.tsv (485.9 MB)
  [Found] test_source3.tsv (482.6 MB)


In [ ]:
# STEP 4: CODE SETUP & REPOSITORY SYNC
print('=' * 70)
print('STEP 4: CODE SETUP & REPOSITORY SYNC')
print('=' * 70)

import os
import subprocess
import sys

WORK_ROOT = '/kaggle/working' if os.path.exists('/kaggle') else os.path.abspath('.')
REPO_DIR = os.path.join(WORK_ROOT, 'amazon-ml')
REPO_URL = 'https://github.com/iam-KamalTripathi/AMC-2026.git'

# 1. If running locally, check if code directory already exists here
local_candidate = os.path.abspath('code/business_entity_resolution')
if os.path.exists(local_candidate) and os.path.exists(os.path.join(local_candidate, 'src')):
    CODE_DIR = local_candidate
    print(f'Using local repository code: {CODE_DIR}')
else:
    # 2. On Kaggle, clone or update from your GitHub repository
    if not os.path.exists(REPO_DIR):
        print(f'Cloning repository from {REPO_URL} into {REPO_DIR}...')
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
    else:
        print(f'Repository already exists at {REPO_DIR}. Pulling latest updates...')
        subprocess.run(['git', '-C', REPO_DIR, 'pull'])
    
    CODE_DIR = os.path.join(REPO_DIR, 'code', 'business_entity_resolution')

assert CODE_DIR and os.path.exists(CODE_DIR), f'ERROR: Code directory not found at {CODE_DIR}'
print(f'Active Code Directory: {CODE_DIR}')

# Add to path & configure CPU workers
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR + ((':' + os.environ['PYTHONPATH']) if 'PYTHONPATH' in os.environ else '')
os.environ['BER_CPUS'] = str(os.cpu_count() or 4)

print(f'PYTHONPATH configured: {CODE_DIR}')
print(f"BER_CPUS configured: {os.environ.get('BER_CPUS')}")


STEP 4: CODE SETUP & PATH CONFIGURATION


AssertionError: ERROR: Could not locate code directory! Checked: ['/kaggle/working/code/business_entity_resolution', '/kaggle/working/code/business_entity_resolution', '/kaggle/working/amazon-ml/code/business_entity_resolution', '/kaggle/working/business_entity_resolution', '/kaggle/working/business_entity_resolution']

In [ ]:
# STEP 4.5: OFFICIAL OUTPUT DIRECTORY & HEADER VERIFIER
print('=' * 70)
print('STEP 4.5: OFFICIAL OUTPUT DIRECTORY & HEADER VERIFIER')
print('=' * 70)

OUT_DIR = os.path.join(WORK_ROOT, 'output')
WORK_DIR = os.path.join(WORK_ROOT, 'work')
LOG_DIR = os.path.join(WORK_ROOT, 'logs')

for d in [OUT_DIR, WORK_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

OFFICIAL_HEADERS = {
    'matching_results.tsv': 'source1_entity_id\tmatched_entity_ids',
    'candidate_pairs.tsv': 'source1_entity_id\tcandidate_entity_ids'
}

def verify_output_headers(out_dir):
    print(f'Checking official TSV outputs in {out_dir}:')
    all_ok = True
    for fname, expected_header in OFFICIAL_HEADERS.items():
        fpath = os.path.join(out_dir, fname)
        if not os.path.exists(fpath):
            print(f'  [PENDING] {fname} not generated yet.')
            all_ok = False
            continue
        with open(fpath, 'r', encoding='utf-8') as f:
            first_line = f.readline().rstrip('\r\n')
        if first_line == expected_header:
            print(f'  [PASS] {fname} header matches exactly: {first_line!r}')
        else:
            print(f'  [FAIL] {fname} header mismatch!\n    Got:      {first_line!r}\n    Expected: {expected_header!r}')
            all_ok = False
    return all_ok

verify_output_headers(OUT_DIR)

In [ ]:
# STEP 5: HYPERPARAMETERS & CONFIGURATION REVIEW
print('=' * 70)
print('STEP 5: HYPERPARAMETERS & CONFIGURATION REVIEW')
print('=' * 70)

DECISION_THRESHOLD = 0.75
BIENC_MAX_PAIRS = 500_000
BIENC_ENC_BS = 2048
XENC_BS = 128
XENC_ENC_BS = 1024

print(f'DATASET_DIR:        {DATASET_DIR}')
print(f'WORK_DIR:           {WORK_DIR}')
print(f'OUT_DIR:            {OUT_DIR}')
print(f'LOG_DIR:            {LOG_DIR}')
print(f'VALIDATOR_PATH:     {VALIDATOR_PATH}')
print('-' * 70)
print(f'Accelerators:       Dual Tesla T4 (GPU T4 x2) + DataParallel enabled')
print(f'CPU Workers:        {os.cpu_count() or 4} vCPUs active for rapidfuzz & Polars')
print(f'Bi-Encoder Pairs:   {BIENC_MAX_PAIRS:,} (convergence achieved in ~12 min)')
print(f'Bi-Encoder Enc BS:  {BIENC_ENC_BS} (1,024 records per GPU in parallel)')
print(f'Cross-Encoder BS:   {XENC_BS} training / {XENC_ENC_BS} scoring ({XENC_ENC_BS//2} per GPU in parallel)')
print(f'Cross-Encoder Mode: --used-only enabled (skips unused train folds 5-9)')
print(f'Decision Threshold: t = {DECISION_THRESHOLD} (calibrated for test decoy distribution)')
print(f'Features Setup:     49 Stage-1 features + 26 active Decoy features (87 total)')
print('=' * 70)

In [ ]:
# PIPELINE EXECUTION ENGINE (Live Streaming Stdout & Precise Timing)
def run_stage(stage_name, module_name, *extra_args):
    print('\n' + '=' * 80)
    print(f'>>> STARTING STAGE: {stage_name} ({module_name})')
    print(f'>>> Time: {time.strftime("%Y-%m-%d %H:%M:%S")}')
    print('=' * 80)
    
    cmd = [
        sys.executable, '-m', module_name,
        '--data-dir', DATASET_DIR,
        '--work-dir', WORK_DIR
    ] + list(extra_args)
    
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['PYTHONPATH'] = CODE_DIR + ':' + env.get('PYTHONPATH', '')
    env['BER_CPUS'] = str(os.cpu_count() or 4)
    
    t_start = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=CODE_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    log_file = os.path.join(LOG_DIR, f'{stage_name}.log')
    with open(log_file, 'w', encoding='utf-8') as lf:
        for line in proc.stdout:
            if 'DeprecationWarning' in line or 'empty_as_null' in line:
                continue
            print(line, end='', flush=True)
            lf.write(line)
            
    ret_code = proc.wait()
    elapsed = time.time() - t_start
    mins, secs = divmod(elapsed, 60)
    
    if ret_code != 0:
        raise RuntimeError(f"Stage '{stage_name}' FAILED with exit code {ret_code}! (Elapsed: {int(mins)}m {secs:.1f}s)")
        
    print(f'>>> COMPLETED STAGE: {stage_name} in {int(mins)}m {secs:.1f}s')
    print('=' * 80)
    return elapsed

stage_timings = {}

In [ ]:
# STAGE 1: PREPARE DATA & NORMALIZATION (4 vCPUs Parallelism)
# Loads TSVs, builds transliteration dictionary, normalizes addresses & localities
stage_timings['1_prepare'] = run_stage('prepare', 'src.prepare')

In [ ]:
# STAGE 2: BI-ENCODER TRAINING & EMBEDDING EXTRACTION (Dual T4 DataParallel)
# Fine-tunes multilingual-e5-small with InfoNCE + hard negatives (500k pairs, FP16)
# Then extracts embeddings for all S1 and query records (train and test) across both GPUs simultaneously
stage_timings['2_biencoder'] = run_stage('bienc', 'src.biencoder', '--max-pairs', str(BIENC_MAX_PAIRS), '--enc-bs', str(BIENC_ENC_BS))

In [ ]:
# STAGE 3: EXACT GPU DENSE kNN RETRIEVAL
# Retrieves query->S1 top-10 and S1->query top-10 within country using FP16 matrix matmuls
stage_timings['3_retrieve'] = run_stage('retrieve', 'src.retrieve')

In [ ]:
# STAGE 4: REFOLD UNMATCHED QUERIES
# Assigns unmatched training queries the fold of the S1 entity they compete with
stage_timings['4_refold'] = run_stage('refold', 'src.refold')

In [ ]:
# STAGE 5: STAGE-1 PAIR FEATURE EXTRACTION (49 FEATURES, 4 vCPUs)
# Computes fuzzy string matching, TF-IDF cosine, Jaro-Winkler, number overlap, embedding ranks
stage_timings['5_features'] = run_stage('features', 'src.features')

In [ ]:
# STAGE 6: LIGHTGBM CANDIDATE PRUNING
# LightGBM ranker on folds 1-4, keeps top <= 4 candidates per query (p >= 0.003)
stage_timings['6_prune'] = run_stage('prune', 'src.ranker', '--stage', 'prune')

In [ ]:
# STAGE 7: DECOY FEATURE EXTRACTION (26 ACTIVE DECOY FEATURES, 4 vCPUs)
# Moved house numbers, added legal tokens, extra-word log-odds, address consistency
stage_timings['7_features2'] = run_stage('feats2', 'src.features2')

In [ ]:
# STAGE 8: CROSS-ENCODER TRAINING & SCORING (Dual T4 DataParallel + --used-only)
# Listwise cross-encoder with 'no-match' slot (FP16, bs=128, enc-bs=1024 across 2 GPUs)
# --used-only skips scoring unused training folds 5-9, cutting cross-encoder time by ~65%
stage_timings['8_crossencoder'] = run_stage('xenc', 'src.crossencoder', '--bs', str(XENC_BS), '--enc-bs', str(XENC_ENC_BS), '--used-only')

In [ ]:
# STAGE 9: FINAL LIGHTGBM RANKER (87 FEATURES)
# Trains LightGBM on 49 stage-1 features + 26 decoy features + cross-encoder logit & margins
stage_timings['9_final'] = run_stage('final', 'src.ranker', '--stage', 'final')

In [ ]:
# STAGE 10: DECISION & TSV OUTPUT GENERATION
# Greedy best-per-query decoding at calibrated threshold t = 0.75
# Generates matching_results.tsv and candidate_pairs.tsv in output/
stage_timings['10_decide'] = run_stage('decide', 'src.decide', '--out-dir', OUT_DIR, '--threshold', str(DECISION_THRESHOLD))

In [ ]:
# STEP 11: VALIDATION & SCORECARD EVALUATION
print('=' * 80)
print('STEP 11: VALIDATION & SCORECARD EVALUATION')
print('=' * 80)

matching_file = os.path.join(OUT_DIR, 'matching_results.tsv')
candidate_file = os.path.join(OUT_DIR, 'candidate_pairs.tsv')
test_dir = os.path.join(DATASET_DIR, 'test') if os.path.exists(os.path.join(DATASET_DIR, 'test')) else DATASET_DIR

# 1. Run official challenge validator
if VALIDATOR_PATH and os.path.exists(VALIDATOR_PATH):
    print(f'Running official validator: {VALIDATOR_PATH}...')
    val_cmd = [
        sys.executable, VALIDATOR_PATH,
        '--matching', matching_file,
        '--candidate', candidate_file,
        '--test-dir', test_dir
    ]
    val_proc = subprocess.run(val_cmd, capture_output=True, text=True)
    print(val_proc.stdout)
    if val_proc.stderr:
        print(val_proc.stderr)
    if val_proc.returncode == 0:
        print('>>> [SUCCESS] Official submission validation PASSED cleanly (exit code 0)!')
    else:
        print(f'>>> [WARNING] Validator returned exit code: {val_proc.returncode}')
else:
    print('Validator script not found in dataset folder; checking headers directly...')
    verify_output_headers(OUT_DIR)

# 2. Evaluate performance on Fold 0
print('\n' + '-' * 80)
print('PERFORMANCE SCORECARD: HOLDOUT VALIDATION FOLD 0')
print('-' * 80)

eval_cmd = [
    sys.executable, '-m', 'src.decide',
    '--data-dir', DATASET_DIR,
    '--work-dir', WORK_DIR,
    '--out-dir', '/tmp/unused',
    '--eval-only'
]
eval_proc = subprocess.run(eval_cmd, cwd=CODE_DIR, capture_output=True, text=True)
for line in eval_proc.stdout.splitlines():
    if any(k in line for k in ['threshold sweep', 'VALIDATION', 'singletons', 'micro', 'country', 't=0.75', '0.750']):
        print(line)

print('-' * 80)
total_sec = sum(stage_timings.values())
print(f'Total Pipeline Execution Time: {total_sec / 60:.1f} minutes ({total_sec / 3600:.2f} hours)')

In [ ]:
# STEP 12: PACKAGE SUBMISSION & DIRECT BROWSER DOWNLOADS
print('=' * 80)
print('STEP 12: PACKAGE SUBMISSION & DIRECT BROWSER DOWNLOADS')
print('=' * 80)

import polars as pl
from IPython.display import FileLink, display, HTML

# 1. Statistics of the generated TSV files
m_df = pl.read_csv(matching_file, separator='\t', infer_schema=False)
c_df = pl.read_csv(candidate_file, separator='\t', infer_schema=False)

n_matched = (m_df['matched_entity_ids'] != '').sum()
n_cand = (c_df['candidate_entity_ids'] != '').sum()

print(f'Matching Results: {m_df.height:,} entities | {n_matched:,} entities with matches')
print(f'Candidate Pairs:  {c_df.height:,} entities | {n_cand:,} entities with candidates')
print(f'File Size matching_results.tsv: {os.path.getsize(matching_file) / 2**20:.2f} MB')
print(f'File Size candidate_pairs.tsv:  {os.path.getsize(candidate_file) / 2**20:.2f} MB')

# 2. Build final submission zip using build_submission_zip.py
build_script = os.path.join(WORK_ROOT, 'amazon-ml', 'build_submission_zip.py')
if not os.path.exists(build_script):
    build_script = os.path.join(WORK_ROOT, 'build_submission_zip.py')
if not os.path.exists(build_script):
    build_script = os.path.abspath('build_submission_zip.py')

team_name = 'final'
doc_file = DOC_TEMPLATE if DOC_TEMPLATE and os.path.exists(DOC_TEMPLATE) else os.path.join(WORK_ROOT, 'amazon-ml', 'Documentation_template.md')
if not os.path.exists(doc_file):
    doc_file = os.path.abspath('Documentation_template.md')

final_zip_path = os.path.join(WORK_ROOT, f'{team_name}_submission.zip')

if os.path.exists(build_script) and os.path.exists(doc_file):
    print(f'\nBuilding official submission package: {final_zip_path}...')
    zip_cmd = [
        sys.executable, build_script,
        '--team', team_name,
        '--matching', matching_file,
        '--candidates', candidate_file,
        '--code-dir', CODE_DIR,
        '--doc', doc_file,
        '--out-dir', WORK_ROOT,
        '--force',
        '--check-subset'
    ]
    zip_res = subprocess.run(zip_cmd, capture_output=True, text=True)
    print(zip_res.stdout)
    if zip_res.stderr:
        print(zip_res.stderr)

# 3. Render direct download links
print('\n' + '=' * 80)
print('BROWSER DOWNLOAD LINKS:')
print('=' * 80)
display(HTML('<h3>Click below to download submission files:</h3>'))

for label, path in [
    ('matching_results.tsv', matching_file),
    ('candidate_pairs.tsv', candidate_file),
    (f'{team_name}_submission.zip', final_zip_path)
]:
    if os.path.exists(path):
        sz = os.path.getsize(path) / 2**20
        print(f'  -> {label} ({sz:.2f} MB)')
        display(FileLink(os.path.relpath(path, start=os.getcwd()) if os.path.exists(os.path.relpath(path, start=os.getcwd())) else path))